In [3]:
import pandas as pd
import numpy as np

# 1. Load the engineered datasets
train_df = pd.read_csv("../data/processed/train_engineered.csv")
test_df = pd.read_csv("../data/processed/test_engineered.csv")

# 2. Separate Features and Target for Train and Test
X_train = train_df.drop(columns=['Churn'])
y_train = train_df['Churn']

X_test = test_df.drop(columns=['Churn'])
y_test = test_df['Churn']

# 3. Define Identifier
id_col = 'customerID'

# 4. Get list of actual feature columns (Excluding customerID)
feature_cols = [col for col in X_train.columns if col != id_col]

# 5. Separate Numerical and Categorical feature names
num_features = X_train[feature_cols].select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X_train[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

# 6. Print detailed summary for engineering validation
print("=== FEATURE SEPARATION SUMMARY ===")
print(f"Total Rows in Train: {X_train.shape[0]}")
print(f"Total Features (excluding ID & Target): {len(feature_cols)}")
print(f"\n--- Numerical Features ({len(num_features)}) ---")
print(num_features)
print(f"\n--- Categorical Features ({len(cat_features)}) ---")
print(cat_features)

=== FEATURE SEPARATION SUMMARY ===
Total Rows in Train: 5634
Total Features (excluding ID & Target): 22

--- Numerical Features (7) ---
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'IsAutomaticPayment', 'MonthlySpendDiff']

--- Categorical Features (15) ---
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\Mohamed\AppData\Local\Temp\ipykernel_15752\2966469165.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X_train[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# 1. Separate strictly binary numericals vs continuous numericals (Optional, but best practice)
# Here we can safely apply StandardScaler to all numerical features.
num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

# 2. Combine into a single Master ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ],
    remainder='drop' # Drops any columns not explicitly mentioned (like customerID)
)

# 3. Fit on X_train ONLY, then Transform both X_train and X_test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 4. Extract Feature Names after One-Hot Encoding for verification
ohe_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(cat_features)
all_transformed_feature_names = list(num_features) + list(ohe_feature_names)

print("=== PIPELINE EXECUTION SUCCESSFUL ===")
print(f"Transformed X_train Shape: {X_train_processed.shape}")
print(f"Transformed X_test Shape:  {X_test_processed.shape}")
print(f"\nTotal Features generated after One-Hot Encoding: {len(all_transformed_feature_names)}")
print("\nFirst 10 transformed feature names:")
print(all_transformed_feature_names[:10])

=== PIPELINE EXECUTION SUCCESSFUL ===
Transformed X_train Shape: (5634, 33)
Transformed X_test Shape:  (1409, 33)

Total Features generated after One-Hot Encoding: 33

First 10 transformed feature names:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'IsAutomaticPayment', 'MonthlySpendDiff', 'gender_Male', 'Partner_Yes', 'Dependents_Yes']


In [5]:
import os
# 1. Sanity Check for NaNs in Transformed Arrays
print("=== SANITY CHECK FOR TRANSFORMED ARRAYS ===")
print("NaNs in Transformed X_train:", np.isnan(X_train_processed).sum())
print("NaNs in Transformed X_test: ", np.isnan(X_test_processed).sum())

# 2. Convert transformed numpy arrays back to DataFrames with proper column names
X_train_final = pd.DataFrame(X_train_processed, columns=all_transformed_feature_names)
X_test_final = pd.DataFrame(X_test_processed, columns=all_transformed_feature_names)

# 3. Add Target column back for clean storage
train_ready = pd.concat([X_train_final, y_train.reset_index(drop=True)], axis=1)
test_ready = pd.concat([X_test_final, y_test.reset_index(drop=True)], axis=1)

=== SANITY CHECK FOR TRANSFORMED ARRAYS ===
NaNs in Transformed X_train: 0
NaNs in Transformed X_test:  0


In [ ]:
# 4. Save Final Modeling-Ready Datasets
train_ready.to_csv("../data/processed/train_ready.csv", index=False)
test_ready.to_csv("../data/processed/test_ready.csv", index=False)

# 5. Save the Pipeline Object for Future Inference (Deployment Readiness)
import joblib
os.makedirs("../models", exist_ok=True)
joblib.dump(preprocessor, "../models/preprocessor_pipeline.pkl")

print("\n=== SUCCESSFUL EXPORT ===")
print("1. 'train_ready.csv' & 'test_ready.csv' saved in 'data/processed/'")
print("2. 'preprocessor_pipeline.pkl' saved in 'models/' directory.")

=== SANITY CHECK FOR TRANSFORMED ARRAYS ===
NaNs in Transformed X_train: 0
NaNs in Transformed X_test:  0

=== SUCCESSFUL EXPORT ===
1. 'train_ready.csv' & 'test_ready.csv' saved in 'data/processed/'
2. 'preprocessor_pipeline.pkl' saved in 'models/' directory.


# Modeling & Evaluation

In [52]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# 1. Separate Features and Target from the ready DataFrames
X_train = train_ready.drop(columns=['Churn']).values
y_train = train_ready['Churn']
y_train = y_train.replace({'No': 0, 'Yes': 1}).astype(int).values

X_test = test_ready.drop(columns=['Churn']).values
y_test = test_ready['Churn']
y_test = y_test.replace({'No': 0, 'Yes': 1}).astype(int).values

In [108]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss

def evaluate_model(model, X, y, model_type='sklearn'):

    if model_type == 'sklearn':
        y_pred_proba = model.predict_proba(X)[:, 1]
        y_pred = (y_pred_proba >= 0.5).astype(int)
    
    else:
        model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X)
            y_pred_proba = model(X_tensor).numpy().flatten()
            y_pred = (y_pred_proba >= 0.5).astype(int)
    
     
    auc_score = roc_auc_score(y, y_pred_proba)
    report = classification_report(y, y_pred)
    accuracy = accuracy_score(y, y_pred)
    cost = log_loss(y, y_pred_proba)
        
    return auc_score, report, accuracy, cost


def print_model_metrics(auc, report, accuracy, cost, split_type):
    print("\n" + "=" * 50)
    print(f"           {split_type.upper()} METRICS")
    print("=" * 50)

    print(f"ROC-AUC       : {auc:.4f}")
    print(f"Accuracy      : {accuracy * 100:.2f}%")
    print(f"Business Cost : {cost:,.2f}")

    print("\nClassification Report:")
    print(report)

    print("=" * 50)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

def train_logistic_regression(X_train, y_train):
    
    print(f"\nTraining Logistic Regression...")
    
    model = LogisticRegression(
        class_weight='balanced',
        solver='liblinear',
        max_iter=1000,
        random_state=42
    )
    
    model.fit(X_train, y_train)

    print(f"Model trained successfully")
    return model


def train_decision_tree(X_train, y_train):
    
    print(f"\nTraining Decision Tree...")
    
    model = DecisionTreeClassifier(
    class_weight='balanced',
    max_depth=5,
    random_state=42
    )
    
    model.fit(X_train, y_train)

    print(f"Model trained successfully")
    return model


def train_random_forest(X_train, y_train):
    
    print(f"\nTraining Random Forest...")
    
    model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=8,
    random_state=42,
    n_jobs=-1
    )
    
    model.fit(X_train, y_train)

    print(f"Model trained successfully")
    return model


In [9]:
# ==========================================
# LOGISTIC REGRESSION BASELINE
# ==========================================

lr_model = train_logistic_regression(X_train, y_train)
lr_auc_score_train, lr_report_train, lr_accuracy_train, lr_cost_train = evaluate_model(lr_model, X_train, y_train, model_type='sklearn')
lr_auc_score_test, lr_report_test, lr_accuracy_test, lr_cost_test = evaluate_model(lr_model, X_test, y_test, model_type='sklearn')

print_model_metrics(
    auc=lr_auc_score_train, 
    report=lr_report_train, 
    accuracy=lr_accuracy_train, 
    cost=lr_cost_train,
    split_type="Logistic Regression - Train"
)

print("\n" + "*" * 70)

print_model_metrics(
    auc=lr_auc_score_test, 
    report=lr_report_test, 
    accuracy=lr_accuracy_test, 
    cost=lr_cost_test,
    split_type="Logistic Regression - Test"
)



Training Logistic Regression...
Model trained successfully

           LOGISTIC REGRESSION - TRAIN METRICS
ROC-AUC       : 0.8517
Accuracy      : 75.81%
Business Cost : 0.48

Classification Report:
              precision    recall  f1-score   support

         0.0       0.91      0.74      0.82      4139
         1.0       0.53      0.80      0.64      1495

    accuracy                           0.76      5634
   macro avg       0.72      0.77      0.73      5634
weighted avg       0.81      0.76      0.77      5634


**********************************************************************

           LOGISTIC REGRESSION - TEST METRICS
ROC-AUC       : 0.8472
Accuracy      : 74.31%
Business Cost : 0.49

Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.73      0.81      1035
         1.0       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71   

In [55]:
# ==========================================
# DECISION TREE BASELINE
# ==========================================

dt_model = train_decision_tree(X_train, y_train)
dt_auc_score_train, dt_report_train, dt_accuracy_train, dt_cost_train = evaluate_model(dt_model, X_train, y_train, model_type='sklearn')
dt_auc_score_test, dt_report_test, dt_accuracy_test, dt_cost_test = evaluate_model(dt_model, X_test, y_test, model_type='sklearn')

print_model_metrics(
    auc=dt_auc_score_train, 
    report=dt_report_train, 
    accuracy=dt_accuracy_train, 
    cost=dt_cost_train,
    split_type="Decision Tree - Train"
)

print("\n" + "*" * 70)

print_model_metrics(
    auc=dt_auc_score_test, 
    report=dt_report_test, 
    accuracy=dt_accuracy_test, 
    cost=dt_cost_test,
    split_type="Decision Tree - Test"
)


Training Decision Tree...
Model trained successfully

           DECISION TREE - TRAIN METRICS
ROC-AUC       : 0.8473
Accuracy      : 74.76%
Business Cost : 0.48

Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.72      0.81      4139
         1.0       0.52      0.82      0.63      1495

    accuracy                           0.75      5634
   macro avg       0.72      0.77      0.72      5634
weighted avg       0.81      0.75      0.76      5634


**********************************************************************

           DECISION TREE - TEST METRICS
ROC-AUC       : 0.8262
Accuracy      : 73.39%
Business Cost : 0.60

Classification Report:
              precision    recall  f1-score   support

         0.0       0.91      0.71      0.80      1035
         1.0       0.50      0.80      0.61       374

    accuracy                           0.73      1409
   macro avg       0.70      0.75      0.71      1409
weighted a

In [56]:
# ==========================================
# RANDOM FOREST BASELINE
# ==========================================

rf_model = train_random_forest(X_train, y_train)
rf_auc_score_train, rf_report_train, rf_accuracy_train, rf_cost_train = evaluate_model(rf_model, X_train, y_train, model_type='sklearn')
rf_auc_score_test, rf_report_test, rf_accuracy_test, rf_cost_test = evaluate_model(rf_model, X_test, y_test, model_type='sklearn')

print_model_metrics(
    auc=rf_auc_score_train, 
    report=rf_report_train, 
    accuracy=rf_accuracy_train, 
    cost=rf_cost_train,
    split_type="Random Forest - Train"
)

print("\n" + "*" * 70)

print_model_metrics(
    auc=rf_auc_score_test, 
    report=rf_report_test, 
    accuracy=rf_accuracy_test, 
    cost=rf_cost_test,
    split_type="Random Forest - Test"
)


Training Random Forest...
Model trained successfully

           RANDOM FOREST - TRAIN METRICS
ROC-AUC       : 0.9042
Accuracy      : 79.77%
Business Cost : 0.42

Classification Report:
              precision    recall  f1-score   support

         0.0       0.94      0.77      0.85      4139
         1.0       0.58      0.86      0.69      1495

    accuracy                           0.80      5634
   macro avg       0.76      0.82      0.77      5634
weighted avg       0.84      0.80      0.81      5634


**********************************************************************

           RANDOM FOREST - TEST METRICS
ROC-AUC       : 0.8418
Accuracy      : 75.23%
Business Cost : 0.48

Classification Report:
              precision    recall  f1-score   support

         0.0       0.91      0.74      0.81      1035
         1.0       0.52      0.79      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted a

In [140]:
# Deep Neural Network Configuration
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 7
BATCH_SIZE = 64
LEARNING_RATE = 0.001

In [127]:
print(f"Class distribution in training: {np.bincount(y_train)}")

Class distribution in training: [4139 1495]


In [141]:
class DeepNN(nn.Module):
    """
    Deep Neural Network for binary classification.
    """
    def __init__(self, input_dim=X_train.shape[1]):
        super(DeepNN, self).__init__()
        self.layer_1 = nn.Linear(input_dim, 4)
        #self.layer_2 = nn.Linear(64, 32)
        #self.layer_3 = nn.Linear(256, 128)
        #self.layer_4 = nn.Linear(128, 64)
        #self.layer_5 = nn.Linear(64, 32)
        self.layer_out = nn.Linear(4, 1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.4)
        self.sigmoid = nn.Sigmoid()

    def forward(self, inputs):
        lay_1 = self.layer_1(inputs)
        lay_1 = self.relu(lay_1)
        lay_1 = self.dropout(lay_1)
        
        '''
        lay_2 = self.layer_2(lay_1)
        lay_2 = self.relu(lay_2)
        lay_2 = self.dropout(lay_2)
        
        lay_3 = self.layer_3(lay_2)
        lay_3 = self.relu(lay_3)
        lay_3 = self.dropout(lay_3)
        
        lay_4 = self.layer_4(lay_3)
        lay_4 = self.relu(lay_4)
        lay_4 = self.dropout(lay_4)
        
        lay_5 = self.layer_5(lay_4)
        lay_5 = self.relu(lay_5)
        lay_5 = self.dropout(lay_5)
        '''
        
        linear_output = self.layer_out(lay_1)
        output = self.sigmoid(linear_output)
        return output


def train_dnn_epoch(model, loader, optimizer, criterion):
    """Train the DNN for one epoch."""
    model.train()
    for _, (X_batch, y_batch) in enumerate(loader):
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        


def train_pytorch_model(model, X_train, y_train, learning_rate=LEARNING_RATE, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """
    Train PyTorch model.
    """
    # Convert to PyTorch tensors
    X_tensor = torch.FloatTensor(X_train)
    y_tensor = torch.FloatTensor(y_train).reshape(-1, 1)

    # Create DataLoader
    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Loss and optimizer
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Training loop
    print(f"Training for {epochs} epochs...")
    for epoch in range(epochs):
        train_dnn_epoch(model, dataloader, optimizer, criterion)
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}/{epochs} completed")

    return model

In [142]:
NN_model = DeepNN()
NN_model = train_pytorch_model(NN_model, X_train, y_train)

Training for 7 epochs...


In [143]:
# Evaluate using your exact functions
nn_auc_train, nn_report_train, nn_acc_train, nn_cost_train = evaluate_model(NN_model, X_train, y_train, model_type='PyTorch')
nn_auc_test, nn_report_test, nn_acc_test, nn_cost_test = evaluate_model(NN_model, X_test, y_test, model_type='PyTorch')


print_model_metrics(
    auc=nn_auc_train, 
    report=nn_report_train, 
    accuracy=nn_acc_train, 
    cost=nn_cost_train,
    split_type="PyTorch NN - Train"
)

print("\n" + "*" * 70)

print_model_metrics(
    auc=nn_auc_test, 
    report=nn_report_test, 
    accuracy=nn_acc_test, 
    cost= nn_cost_test,
    split_type="PyTorch NN - Test"
)


           PYTORCH NN - TRAIN METRICS
ROC-AUC       : 0.8390
Accuracy      : 80.07%
Business Cost : 0.43

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      4139
           1       0.67      0.49      0.57      1495

    accuracy                           0.80      5634
   macro avg       0.75      0.70      0.72      5634
weighted avg       0.79      0.80      0.79      5634


**********************************************************************

           PYTORCH NN - TEST METRICS
ROC-AUC       : 0.8373
Accuracy      : 80.20%
Business Cost : 0.43

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.67      0.51      0.58       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [144]:
import os

# Create relative path to save inside the project's "models" folder
save_dir = "../models"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "churn_nn_weights.pth")

# Save the PyTorch model state_dict
torch.save(NN_model.state_dict(), save_path)

print(f"Final model weights saved to: {save_path}")

Final model weights saved to: ../models\churn_nn_weights.pth
